# 🎤 AISinger — make any voice sing any song

<a href="https://colab.research.google.com/github/FightWithCode/AISinger/blob/main/notebooks/AISinger_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).

How to use it: run **Step 1** every session. Then use whichever sections you need:

| Section | What it does |
|---|---|
| **A. Clone a voice (train)** | Train an RVC model of *your* voice (or any singer) from 10–30 min of audio. Best quality. |
| **B. Instant voice (zero-shot)** | Clone a voice from one 10–25 s clip with Seed-VC. No training. |
| **C. Import a voice model** | Use an existing RVC model (`.pth` + `.index`, zip or link). |
| **D. Make a cover** | Song in → song sung by the chosen voice out. |
| **E. Convert a vocal** | Convert a vocal-only recording (e.g. you singing) into another voice. |
| **F. Speak text** | Text-to-speech in a cloned voice. |

Your voices, datasets and outputs are saved to **Google Drive → `MyDrive/AISinger`**, so they survive disconnects.

⚠️ Only clone voices you have the right to use. Covers of famous singers are fine for personal use; publishing them may infringe personality rights and copyright.

In [ ]:
#@title 1. Setup (run every session) { display-mode: "form" }
REPO_URL = "https://github.com/FightWithCode/AISinger"  #@param {type:"string"}
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
ENGINES = "all"  #@param ["all", "rvc", "seedvc"]

import os, subprocess, sys
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE — switch to a GPU runtime (Runtime → Change runtime type)!")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["AISINGER_HOME"] = "/content/drive/MyDrive/AISinger"
else:
    os.environ["AISINGER_HOME"] = "/content/AISinger"
os.environ["AISINGER_CACHE"] = "/content/aisinger-cache"
HOME = Path(os.environ["AISINGER_HOME"])
UPLOADS = HOME / "uploads"
UPLOADS.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", f"aisinger[gpu] @ git+{REPO_URL}"], check=True)

def aisinger(*args):
    """Run an aisinger command and stream its output."""
    p = subprocess.Popen(["aisinger", *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("aisinger failed — see the log above.")

def get_files(path_in_drive=""):
    """Use a path in Drive / Colab if given, otherwise open an upload dialog."""
    if path_in_drive.strip():
        p = Path(path_in_drive.strip())
        if not p.exists():
            raise FileNotFoundError(p)
        return [str(p)]
    from google.colab import files
    saved = []
    for name, data in files.upload().items():
        dst = UPLOADS / name
        dst.write_bytes(data)
        Path(name).unlink(missing_ok=True)
        saved.append(str(dst))
    return saved

def play(path):
    from IPython.display import Audio, display
    display(Audio(str(path)))
    print("Saved at:", path)

aisinger("setup", "--engine", ENGINES)
aisinger("doctor")
aisinger("voices")

---
## A. Clone a voice (train an RVC model)

**Recording tips for *your* voice** (this matters more than any setting):
- 10–30 minutes of you **singing** (plus some speaking is fine), with no music, in a quiet, non-echoey room.
- Cover your full range: low, high, soft, loud, held notes.
- Any format is fine (wav/mp3/m4a). Several files are fine. Put them in a Drive folder or upload them.

**To clone a singer** (e.g. Kishore Kumar), use 15–30 songs and tick `FULL_SONGS`. The vocals are extracted, chorus removed, and echo/reverb removed automatically. Solo songs with few instruments work best.

In [ ]:
#@title A1. Prepare the dataset
VOICE_NAME = "me"  #@param {type:"string"}
#@markdown Folder or file in Drive (e.g. `/content/drive/MyDrive/my_recordings`). Leave empty to upload.
SOURCE_PATH = ""  #@param {type:"string"}
FULL_SONGS = False  #@param {type:"boolean"}
REMOVE_REVERB = False  #@param {type:"boolean"}
REMOVE_NOISE = False  #@param {type:"boolean"}

args = ["dataset", VOICE_NAME, *get_files(SOURCE_PATH)]
if FULL_SONGS:
    args += ["--isolate", "--lead-only", "--dereverb"]
elif REMOVE_REVERB:
    args += ["--dereverb"]
if REMOVE_NOISE:
    args += ["--denoise"]
aisinger(*args)

In [ ]:
#@title A2. Train  (≈20–60 min on a T4; resumes automatically if Colab disconnects)
VOICE_NAME = "me"  #@param {type:"string"}
EPOCHS = 300  #@param {type:"slider", min:50, max:1000, step:25}
SAVE_EVERY = 25  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
START_OVER = False  #@param {type:"boolean"}

args = ["train", VOICE_NAME, "--epochs", EPOCHS, "--save-every", SAVE_EVERY, "--batch-size", BATCH_SIZE]
if START_OVER:
    args.append("--fresh")
aisinger(*args)
aisinger("use", VOICE_NAME)

In [ ]:
#@title A3. (Optional) Use an earlier epoch if the latest one sounds harsh or robotic
VOICE_NAME = "me"  #@param {type:"string"}
#@markdown Copy a file name from the list printed by A2 (e.g. `me_200e_5000s.pth`). Leave empty to use the latest.
MODEL_FILE = ""  #@param {type:"string"}
aisinger("use", VOICE_NAME, *( [MODEL_FILE] if MODEL_FILE else [] ))

---
## B. Instant voice from one clip (zero-shot, Seed-VC)
No training: give it **10–25 seconds** of the target voice. Clean a-cappella audio works best. If the clip has music, tick `CLIP_HAS_MUSIC`.
Use it in D/E with `ENGINE = "seedvc"`. For the most natural results, train an RVC model (section A) instead.

In [ ]:
#@title B1. Set the reference clip
VOICE_NAME = "kishore_zs"  #@param {type:"string"}
CLIP_PATH = ""  #@param {type:"string"}
CLIP_HAS_MUSIC = True  #@param {type:"boolean"}
START_AT_SECONDS = 0  #@param {type:"number"}
args = ["reference", VOICE_NAME, get_files(CLIP_PATH)[0], "--start", START_AT_SECONDS]
if CLIP_HAS_MUSIC:
    args.append("--isolate")
aisinger(*args)
play(HOME / "voices" / VOICE_NAME / "reference.wav")

In [ ]:
#@title B2. (Optional) Fine-tune Seed-VC on a dataset prepared in A1 (≈15–30 min)
VOICE_NAME = "kishore_zs"  #@param {type:"string"}
DATASET_NAME = "kishore"  #@param {type:"string"}
STEPS = 1000  #@param {type:"integer"}
aisinger("train", VOICE_NAME, "--engine", "seedvc", "--data", HOME / "datasets" / DATASET_NAME, "--steps", STEPS)

---
## C. Import an existing RVC voice model
Paste a direct link (HuggingFace / Google Drive zip) or a Drive path to a `.pth`, folder or `.zip`.

In [ ]:
#@title C. Import
VOICE_NAME = "kishore"  #@param {type:"string"}
MODEL_LINK_OR_PATH = ""  #@param {type:"string"}
aisinger("import", VOICE_NAME, MODEL_LINK_OR_PATH)
aisinger("voices")

---
## D. Make a cover 🎶
- `PITCH`: shifts **only the vocal**. Use `+12` for male→female or `-12` for female→male. This keeps the song's key.
- `KEY`: transposes **the whole song**, vocal and music. Use it when the original key is too high or low for the new singer (e.g. `-2`).
- `BACKING`: what to do with chorus/backing vocals: keep the originals, convert them too, or drop them.

In [ ]:
#@title D. Cover
#@markdown Song file in Drive (leave empty to upload):
SONG_PATH = ""  #@param {type:"string"}
VOICE = "me"  #@param {type:"string"}
ENGINE = "rvc"  #@param ["rvc", "seedvc"]
PITCH = 0  #@param {type:"slider", min:-24, max:24, step:1}
KEY = 0  #@param {type:"slider", min:-6, max:6, step:1}
BACKING = "keep"  #@param ["keep", "convert", "drop"]
REVERB = 0.25  #@param {type:"slider", min:0, max:1, step:0.05}
VOCAL_GAIN_DB = 0  #@param {type:"slider", min:-6, max:6, step:0.5}
#@markdown **RVC fine-tuning** (defaults are good; raise `INDEX_RATE` for more of the target's accent/tone, lower `PROTECT` if breaths/"s" sounds get robotic):
INDEX_RATE = 0.5  #@param {type:"slider", min:0, max:1, step:0.05}
PROTECT = 0.33  #@param {type:"slider", min:0, max:0.5, step:0.01}
F0_METHOD = "rmvpe"  #@param ["rmvpe", "fcpe", "crepe", "hybrid[crepe+rmvpe]"]
#@markdown **Seed-VC**: more steps = better and slower
SEEDVC_STEPS = 40  #@param {type:"slider", min:10, max:100, step:5}

from aisinger.utils import slugify
song = get_files(SONG_PATH)[0]
aisinger("cover", song, "--voice", VOICE, "--engine", ENGINE, "--pitch", PITCH, "--key", KEY,
         "--backing", BACKING, "--reverb", REVERB, "--vocal-gain", VOCAL_GAIN_DB,
         "--index-rate", INDEX_RATE, "--protect", PROTECT, "--f0-method", F0_METHOD, "--steps", SEEDVC_STEPS)
outs = sorted((HOME / "outputs" / slugify(Path(song).stem)).glob("*.mp3"), key=lambda f: f.stat().st_mtime)
play(outs[-1])

---
## E. Convert a vocal-only recording
For example, sing a song yourself (a-cappella) and convert it to another voice. This often sounds **more natural** than converting the original singer, because your timing and expression stay intact.

In [ ]:
#@title E. Convert
VOCAL_PATH = ""  #@param {type:"string"}
VOICE = "kishore"  #@param {type:"string"}
ENGINE = "rvc"  #@param ["rvc", "seedvc"]
PITCH = 0  #@param {type:"slider", min:-24, max:24, step:1}
REMOVE_ROOM_REVERB = False  #@param {type:"boolean"}
src = get_files(VOCAL_PATH)[0]
out = HOME / "outputs" / "converted" / (Path(src).stem + f"__{VOICE}.wav")
args = ["convert", src, "--voice", VOICE, "--engine", ENGINE, "--pitch", PITCH, "-o", out]
if REMOVE_ROOM_REVERB:
    args.append("--dereverb")
aisinger(*args)
play(out)

---
## F. Speak text in a cloned voice (RVC voices)
Base voices: `hi-IN-MadhurNeural` (Hindi male), `hi-IN-SwaraNeural` (Hindi female), `en-IN-PrabhatNeural`, `en-IN-NeerjaNeural`, `en-US-GuyNeural`, …
Pick a base voice of the same gender as the target, or adjust `PITCH`.

In [ ]:
#@title F. Speak
TEXT = "Namaste! Yeh meri awaaz mein bola gaya hai."  #@param {type:"string"}
VOICE = "me"  #@param {type:"string"}
TTS_VOICE = "hi-IN-MadhurNeural"  #@param {type:"string"}
PITCH = 0  #@param {type:"integer"}
out = HOME / "outputs" / "speech" / f"{VOICE}_speech.wav"
aisinger("speak", TEXT, "--voice", VOICE, "--tts-voice", TTS_VOICE, "--pitch", PITCH, "-o", out)
play(out)